# DLC Spatial Analysis (Whole Session)

Pixel-bin occupancy maps, 2D heatmaps, and difference maps for DLC
bodypart tracking across whole sessions.

Outputs go to `stroke_orofacial_pipeline_outputs/dlc_analysis/spatial_whole_session/`
on MICROSCOPE.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None for all dates in session range.
DATES = None

# Bodyparts to analyze
BODYPARTS = ["nose", "jaw", "tongue"]

# Centering modes (each bodypart analyzed under each mode)
CENTER_MODES = ["nose", "spout_midpoint", "eye_midpoint"]

# Overwrite mode: "skip" (default) or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging
from pathlib import Path

import numpy as np

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.dlc_io.loaders import discover_dlc_filtered_files
from stroke_orofacial.dlc_io.parquet import read_centered_parquet, _param_sig
from stroke_orofacial.dlc_spatial.pixel_bins import (
    build_histogram, build_stats_grid, build_baseline_histogram,
)
from stroke_orofacial.dlc_spatial.heatmaps import plot_heatmap, save_heatmap_png
from stroke_orofacial.dlc_spatial.difference_maps import (
    compute_difference_map, plot_difference_map, save_difference_map_png,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
spatial_config = defaults["dlc_spatial"]
centering_config = defaults.get("dlc_centering", {})
output_root = resolver.pipeline_outputs("dlc_analysis") / "spatial_whole_session"

lik_thr_centering = centering_config.get("likelihood_threshold", 0.7)

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for center_mode in CENTER_MODES:
        sig = _param_sig(center_mode, lik_thr_centering)

        print(f"\n{'='*60}")
        print(f"{animal_id} | {center_mode} | {len(dates)} dates x {len(BODYPARTS)} bodyparts")
        print(f"{'='*60}")

        for bodypart in BODYPARTS:
            n_ok, n_skip, n_err = 0, 0, 0

            for date in dates:
                date_str = date.strftime("%Y%m%d")

                # Find the centered parquet
                parquet_dir = (
                    resolver.pipeline_outputs("dlc_analysis")
                    / f"centered_{center_mode}" / animal_id
                )
                dlc_dir = resolver.dlc_output_dir(animal_id)
                h5_files = discover_dlc_filtered_files(dlc_dir, date_str)
                if not h5_files:
                    n_err += 1
                    continue

                parquet_name = f"{h5_files[0].name}.intermediate.{sig}.parquet"
                parquet_path = parquet_dir / parquet_name
                if not parquet_path.exists():
                    n_err += 1
                    continue

                # Output paths
                out_dir = output_root / f"{bodypart}_{center_mode}" / animal_id
                heatmap_path = out_dir / f"heatmap_{date_str}.png"

                if OVERWRITE == "skip" and heatmap_path.exists():
                    n_skip += 1
                    continue

                try:
                    df = read_centered_parquet(parquet_path)

                    # Heatmap
                    fig = plot_heatmap(
                        df, bodypart, spatial_config,
                        title=f"{animal_id} {bodypart} {date_str} ({center_mode})",
                    )
                    save_heatmap_png(fig, heatmap_path, dpi=spatial_config.get("save_dpi", 200))

                    # Stats grid NPZ
                    grid = build_stats_grid(df, bodypart, spatial_config)
                    if grid is not None:
                        npz_path = out_dir / f"stats_grid_{date_str}.npz"
                        npz_path.parent.mkdir(parents=True, exist_ok=True)
                        np.savez(str(npz_path), **grid)

                    n_ok += 1
                except Exception as e:
                    n_err += 1
                    logging.warning("%s/%s/%s/%s: %s", animal_id, date_str, bodypart, center_mode, e)

            print(f"  {bodypart}: ok={n_ok} skip={n_skip} err={n_err}")